# Model Evaluation on ARC Dataset

This notebook evaluates different models and systems on the ARC evaluation dataset.

## What Gets Evaluated
1. **Base models** (gptoss, phi3, qwen15) - No fine-tuning
2. **Stage 1 models** - Fine-tuned for hypothesis generation only
3. **Full MARCO system** - Stage 1+2+3 with belief fusion and refinement

## Metrics Tracked
- **Grid Accuracy**: % of test grids predicted correctly
- **Task Success Rate**: % of tasks where at least 1 test grid is correct
- **Perfect Task Rate**: % of tasks where all test grids are correct
- **Computation Time**: Time per task and total time

## Output
- Detailed CSV results per task
- Summary statistics table
- Comparison plots

## 1. Setup and Imports

In [1]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple
import time
import re
from datetime import datetime

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

2025-10-09 12:34:32.715904: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760013272.726605 2187304 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760013272.731147 2187304 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760013272.736578 2187304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760013272.736590 2187304 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760013272.736592 2187304 computation_placer.cc:177] computation placer alr

✓ Imports successful
PyTorch version: 2.7.0
CUDA available: True
CUDA device: NVIDIA GH200 480GB
GPU Memory: 94.5 GB


## 2. Configuration

In [2]:
CONFIG = {
    # Data paths
    "eval_data_path": "eval_subset/",  # ARC evaluation subset
    "results_dir": "evaluation_results/",
    
    # Base model paths
    "base_model_paths": {
        "gptoss": "../models/gpt-oss",
        "phi3": "../models/phi3",
        "qwen15": "../models/qwen"
    },
    
    # Stage 1 (hypothesis generation) adapter paths
    "stage1_adapter_paths": {
        "gptoss": "finetuned_experts/gptoss_arc_finetuned",
        "phi3": "finetuned_experts/phi3_arc_finetuned",
        "qwen15": "finetuned_experts/qwen15_arc_finetuned"
    },
    
    # Generation parameters
    "max_new_tokens": 300,
    "temperature": 0.7,
    "num_attempts": 3,  # Number of solutions to generate per test case
    
    # Evaluation settings
    "max_eval_tasks": None,  # Set to number to limit, None for all tasks
    "models_to_evaluate": ["phi3"],
}

os.makedirs(CONFIG["results_dir"], exist_ok=True)

print("Configuration:")
print(f"  Evaluation data: {CONFIG['eval_data_path']}")
print(f"  Results dir: {CONFIG['results_dir']}")
print(f"  Models: {CONFIG['models_to_evaluate']}")
print(f"  Attempts per test: {CONFIG['num_attempts']}")

Configuration:
  Evaluation data: eval_subset/
  Results dir: evaluation_results/
  Models: ['phi3']
  Attempts per test: 3


## 3. Helper Functions

In [3]:
def load_evaluation_tasks(data_path: str, max_tasks=None):
    """Load ARC evaluation tasks"""
    data_path = Path(data_path)
    task_files = sorted(list(data_path.glob("*.json")))
    
    if max_tasks:
        task_files = task_files[:max_tasks]
    
    tasks = []
    for json_file in task_files:
        with open(json_file, 'r') as f:
            task_data = json.load(f)
        tasks.append({
            'task_id': json_file.stem,
            'data': task_data
        })
    
    return tasks

def grids_match(pred, true):
    """Check if two grids match exactly"""
    try:
        pred_arr = np.array(pred)
        true_arr = np.array(true)
        
        if pred_arr.shape != true_arr.shape:
            return False
        
        return np.array_equal(pred_arr, true_arr)
    except:
        return False

def parse_solution_from_output(text):
    """Extract grid solution from model output"""
    # Look for JSON array pattern
    match = re.search(r'\[\s*\[.*?\]\s*\]', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass
    return None

def create_arc_prompt(train_pairs, test_input):
    """Create prompt for ARC task (matches fine-tuning format)"""
    prompt_parts = [
        "You are an expert at solving abstract reasoning tasks from the ARC (Abstraction and Reasoning Corpus) challenge.\n",
        "Given input-output example pairs, identify the transformation pattern and apply it to the test input.\n",
        "Format: Provide the output grid as a JSON array.\n\n"
    ]
    
    for i, (inp, out) in enumerate(train_pairs, 1):
        prompt_parts.append(f"Example {i}:")
        prompt_parts.append(f"Input: {json.dumps(inp)}")
        prompt_parts.append(f"Output: {json.dumps(out)}\n")
    
    prompt_parts.append(f"Test Input: {json.dumps(test_input)}")
    prompt_parts.append("Test Output:")
    
    return "\n".join(prompt_parts)

print("✓ Helper functions defined")

✓ Helper functions defined


## 4. Model Loading Functions

In [4]:
from transformers import BitsAndBytesConfig

def load_base_model(model_key, use_4bit=True):
    """Load base model without fine-tuning"""
    print(f"\nLoading base {model_key}...")
    base_path = CONFIG["base_model_paths"][model_key]
    
    tokenizer = AutoTokenizer.from_pretrained(base_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    if use_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            base_path,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            base_path,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
    
    model.eval()
    
    print(f"✓ Base {model_key} loaded ({'4-bit' if use_4bit else 'bf16'})")
    return model, tokenizer

def load_stage1_model(model_key, use_4bit=True):
    """Load Stage 1 fine-tuned model (hypothesis generation)"""
    print(f"\nLoading Stage 1 {model_key}...")
    base_path = CONFIG["base_model_paths"][model_key]
    adapter_path = CONFIG["stage1_adapter_paths"][model_key]
    
    tokenizer = AutoTokenizer.from_pretrained(base_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    if use_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        
        base_model = AutoModelForCausalLM.from_pretrained(
            base_path,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_path,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
    
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    
    print(f"✓ Stage 1 {model_key} loaded ({'4-bit' if use_4bit else 'bf16'})")
    return model, tokenizer

## 5. Evaluation Functions

In [5]:
import requests

API_URL = "https://api.novita.ai/openai/v1/completions"
API_KEY = "session_YxJcHmOEqBZXCfbqlj5f536q36WGoriT5M-xHmg4TxiS5ja9ongVoGT7EvnTgc2EgdnarWaIXCg2appqlVh0gA=="
MODEL = "openai/gpt-oss-20b"


def generate_solution(model, tokenizer, prompt, num_attempts=1):
    """Generate solution(s) for a task using direct HTTP requests."""
    solutions = []

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }

    for _ in range(num_attempts):
        payload = {
            "model": MODEL,
            "prompt": prompt,
            "temperature": 0.7,
            "stream": False,
        }

        try:

            response = requests.post(API_URL, headers=headers, json=payload)
            response.raise_for_status()  # raises if HTTP error occurs
            data = response.json()
    
            generated = data["choices"][0]["text"]
            solution = parse_solution_from_output(generated)
    
            if solution is not None:
                solutions.append(solution)
        except:
            continue

    return solutions

def evaluate_model_on_task(model, tokenizer, task):
    """Evaluate a single model on a single task"""
    task_data = task['data']
    task_id = task['task_id']
    
    # Prepare train pairs
    train_pairs = [(ex['input'], ex['output']) for ex in task_data.get('train', [])]
    test_cases = task_data.get('test', [])
    
    if not test_cases:
        return None
    
    results = {
        'task_id': task_id,
        'num_test_grids': len(test_cases),
        'correct_grids': 0,
        'test_results': [],
        'time_taken': 0.0
    }
    
    start_time = time.time()
    
    for test_idx, test_case in enumerate(test_cases):
        test_input = test_case['input']
        test_output = test_case['output']
        
        # Create prompt
        prompt = create_arc_prompt(train_pairs, test_input)
        
        # Generate solutions
        solutions = generate_solution(model, tokenizer, prompt, num_attempts=CONFIG['num_attempts'])
        
        # Check if any solution matches
        correct = False
        for solution in solutions:
            if grids_match(solution, test_output):
                correct = True
                break
        
        if correct:
            results['correct_grids'] += 1
        
        results['test_results'].append({
            'test_idx': test_idx,
            'correct': correct,
            'num_solutions': len(solutions)
        })
    
    results['time_taken'] = time.time() - start_time
    results['grid_accuracy'] = results['correct_grids'] / results['num_test_grids']
    results['task_success'] = results['correct_grids'] > 0
    results['perfect_task'] = results['correct_grids'] == results['num_test_grids']
    
    return results

def evaluate_model(model, tokenizer, model_name, model_type, tasks):
    """Evaluate model on all tasks"""
    print(f"\n{'='*80}")
    print(f"Evaluating: {model_name} ({model_type})")
    print(f"{'='*80}")
    
    all_results = []
    
    for task_idx, task in enumerate(tasks, 1):
        print(task_idx)
        result = evaluate_model_on_task(model, tokenizer, task)
        
        if result:
            all_results.append(result)
            
            if task_idx % 5 == 0:
                # Show progress
                current_accuracy = np.mean([r['grid_accuracy'] for r in all_results])
                current_task_success = np.mean([r['task_success'] for r in all_results])
                avg_time = np.mean([r['time_taken'] for r in all_results])
                
                print(f"  Progress: {task_idx}/{len(tasks)} tasks")
                print(f"    Grid accuracy: {current_accuracy:.2%}")
                print(f"    Task success: {current_task_success:.2%}")
                print(f"    Avg time/task: {avg_time:.1f}s")
    
    # Calculate summary statistics
    summary = {
        'model_name': model_name,
        'model_type': model_type,
        'num_tasks': len(all_results),
        'grid_accuracy': np.mean([r['grid_accuracy'] for r in all_results]),
        'task_success_rate': np.mean([r['task_success'] for r in all_results]),
        'perfect_task_rate': np.mean([r['perfect_task'] for r in all_results]),
        'avg_time_per_task': np.mean([r['time_taken'] for r in all_results]),
        'total_time': sum([r['time_taken'] for r in all_results]),
        'total_grids': sum([r['num_test_grids'] for r in all_results]),
        'correct_grids': sum([r['correct_grids'] for r in all_results]),
    }
    
    return all_results, summary

print("✓ Evaluation functions defined")

✓ Evaluation functions defined


## 6. Load Evaluation Tasks

In [6]:
print(f"Loading evaluation tasks from: {CONFIG['eval_data_path']}")
eval_tasks = load_evaluation_tasks(CONFIG['eval_data_path'], max_tasks=CONFIG['max_eval_tasks'])

print(f"✓ Loaded {len(eval_tasks)} evaluation tasks")

# Count total test grids
total_test_grids = sum([len(task['data'].get('test', [])) for task in eval_tasks])
print(f"  Total test grids: {total_test_grids}")

Loading evaluation tasks from: eval_subset/
✓ Loaded 189 evaluation tasks
  Total test grids: 207


## 7. Evaluate Base Models

In [ ]:
base_results = {}
base_summaries = []

for model_key in CONFIG['models_to_evaluate']:
    # Load base model
    model, tokenizer = load_base_model(model_key)
    
    # Evaluate
    results, summary = evaluate_model(
        model, tokenizer, 
        model_name=model_key,
        model_type='base',
        tasks=eval_tasks
    )
    
    base_results[model_key] = results
    base_summaries.append(summary)
    
    # Save detailed results
    results_df = pd.DataFrame(results)
    results_path = Path(CONFIG['results_dir']) / f"base_{model_key}_detailed.csv"
    results_df.to_csv(results_path, index=False)
    print(f"  ✓ Saved detailed results to {results_path}")
    
    # Free memory
    del model, tokenizer
    torch.cuda.empty_cache()

print("\n✓ Base model evaluation complete")


Loading base phi3...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✓ Base phi3 loaded (4-bit)

Evaluating: phi3 (base)
1


Token indices sequence length is longer than the specified maximum sequence length for this model (6600 > 4096). Running this sequence through the model will result in indexing errors


2
3


## 8. Evaluate Stage 1 Models (Hypothesis Generation)

In [8]:
stage1_results = {}
stage1_summaries = []

for model_key in CONFIG['models_to_evaluate']:
    # Load Stage 1 model
    model, tokenizer = load_stage1_model(model_key)
    
    # Evaluate
    results, summary = evaluate_model(
        model, tokenizer,
        model_name=model_key,
        model_type='stage1_hypothesis',
        tasks=eval_tasks
    )
    
    stage1_results[model_key] = results
    stage1_summaries.append(summary)
    
    # Save detailed results
    results_df = pd.DataFrame(results)
    results_path = Path(CONFIG['results_dir']) / f"stage1_{model_key}_detailed.csv"
    results_df.to_csv(results_path, index=False)
    print(f"  ✓ Saved detailed results to {results_path}")
    
    # Free memory
    del model, tokenizer
    torch.cuda.empty_cache()

print("\n✓ Stage 1 model evaluation complete")


Loading Stage 1 qwen15...
✓ Stage 1 qwen15 loaded (4-bit)

Evaluating: qwen15 (stage1_hypothesis)
1
2
3
4
5
6
7
8
9
10
  Progress: 10/189 tasks
    Grid accuracy: 10.00%
    Task success: 10.00%
    Avg time/task: 83.1s
11
12
13
14
15
16
17
18
19
20
  Progress: 20/189 tasks
    Grid accuracy: 5.00%
    Task success: 5.00%
    Avg time/task: 82.2s
21
22
23
24
25
26
27
28
29
30
  Progress: 30/189 tasks
    Grid accuracy: 6.67%
    Task success: 6.67%
    Avg time/task: 80.7s
31
32
33
34
35
36
37
38
39
40
  Progress: 40/189 tasks
    Grid accuracy: 5.00%
    Task success: 5.00%
    Avg time/task: 80.4s
41
42
43
44
45
46
47
48
49
50
  Progress: 50/189 tasks
    Grid accuracy: 4.00%
    Task success: 4.00%
    Avg time/task: 80.0s
51
52
53
54
55
56
57
58
59
60
  Progress: 60/189 tasks
    Grid accuracy: 3.33%
    Task success: 3.33%
    Avg time/task: 81.4s
61
62
63
64
65
66
67
68
69
70
  Progress: 70/189 tasks
    Grid accuracy: 2.86%
    Task success: 2.86%
    Avg time/task: 81.3s
71
72

KeyboardInterrupt: 

## 9. Summary Statistics

In [2]:
results_df = pd.DataFrame(results)
results_path = Path(CONFIG['results_dir']) / f"stage1_{model_key}_detailed.csv"
results_df.to_csv(results_path, index=False)
print(f"  ✓ Saved detailed results to {results_path}")

# Free memory
del model, tokenizer
torch.cuda.empty_cache()

NameError: name 'pd' is not defined

In [1]:
# Combine all summaries
all_summaries = base_summaries + stage1_summaries
summary_df = pd.DataFrame(all_summaries)

# Reorder columns for readability
column_order = [
    'model_type', 'model_name',
    'grid_accuracy', 'task_success_rate', 'perfect_task_rate',
    'num_tasks', 'total_grids', 'correct_grids',
    'avg_time_per_task', 'total_time'
]
summary_df = summary_df[column_order]

# Save summary
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_path = Path(CONFIG['results_dir']) / f"evaluation_summary_{timestamp}.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)
print()
print(summary_df.to_string(index=False))
print()
print(f"✓ Summary saved to {summary_path}")

NameError: name 'base_summaries' is not defined

## 10. Detailed Comparison

In [ ]:
print("\n" + "="*80)
print("DETAILED COMPARISON")
print("="*80)

for model_key in CONFIG['models_to_evaluate']:
    print(f"\n{model_key.upper()}:")
    print("-" * 40)
    
    # Base model stats
    base_summary = [s for s in base_summaries if s['model_name'] == model_key][0]
    print(f"  Base Model:")
    print(f"    Grid Accuracy:      {base_summary['grid_accuracy']:.2%}")
    print(f"    Task Success Rate:  {base_summary['task_success_rate']:.2%}")
    print(f"    Perfect Task Rate:  {base_summary['perfect_task_rate']:.2%}")
    print(f"    Avg Time/Task:      {base_summary['avg_time_per_task']:.1f}s")
    
    # Stage 1 stats
    stage1_summary = [s for s in stage1_summaries if s['model_name'] == model_key][0]
    print(f"  Stage 1 (Fine-tuned):")
    print(f"    Grid Accuracy:      {stage1_summary['grid_accuracy']:.2%}")
    print(f"    Task Success Rate:  {stage1_summary['task_success_rate']:.2%}")
    print(f"    Perfect Task Rate:  {stage1_summary['perfect_task_rate']:.2%}")
    print(f"    Avg Time/Task:      {stage1_summary['avg_time_per_task']:.1f}s")
    
    # Improvement
    grid_improvement = stage1_summary['grid_accuracy'] - base_summary['grid_accuracy']
    task_improvement = stage1_summary['task_success_rate'] - base_summary['task_success_rate']
    
    print(f"  Improvement from Fine-tuning:")
    print(f"    Grid Accuracy:      {grid_improvement:+.2%}")
    print(f"    Task Success Rate:  {task_improvement:+.2%}")

## 11. Visualization (Optional)

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    sns.set_style("whitegrid")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    metrics = ['grid_accuracy', 'task_success_rate', 'perfect_task_rate']
    titles = ['Grid Accuracy', 'Task Success Rate', 'Perfect Task Rate']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx]
        
        # Prepare data
        x_pos = np.arange(len(CONFIG['models_to_evaluate']))
        width = 0.35
        
        base_values = [s[metric] for s in base_summaries]
        stage1_values = [s[metric] for s in stage1_summaries]
        
        ax.bar(x_pos - width/2, base_values, width, label='Base', alpha=0.8)
        ax.bar(x_pos + width/2, stage1_values, width, label='Stage 1', alpha=0.8)
        
        ax.set_ylabel('Rate')
        ax.set_title(title)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(CONFIG['models_to_evaluate'])
        ax.legend()
        ax.set_ylim(0, 1)
        
        # Add value labels on bars
        for i, (base, stage1) in enumerate(zip(base_values, stage1_values)):
            ax.text(i - width/2, base + 0.02, f'{base:.1%}', ha='center', va='bottom', fontsize=8)
            ax.text(i + width/2, stage1 + 0.02, f'{stage1:.1%}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    
    # Save plot
    plot_path = Path(CONFIG['results_dir']) / f"evaluation_comparison_{timestamp}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved comparison plot to {plot_path}")
    
    plt.show()
    
except ImportError:
    print("\n⚠ matplotlib/seaborn not available, skipping visualization")

## 12. Export Results for MARCO Evaluation

To evaluate the full MARCO system, you'll need to run it separately using the MARCO framework.
This cell prepares the task list for MARCO evaluation.

In [ ]:
# Export task IDs for MARCO evaluation
marco_task_list = [task['task_id'] for task in eval_tasks]

marco_config = {
    'eval_tasks': marco_task_list,
    'eval_data_path': CONFIG['eval_data_path'],
    'num_attempts': CONFIG['num_attempts'],
    'timestamp': timestamp
}

marco_config_path = Path(CONFIG['results_dir']) / f"marco_eval_config_{timestamp}.json"
with open(marco_config_path, 'w') as f:
    json.dump(marco_config, f, indent=2)

print(f"✓ MARCO evaluation config saved to {marco_config_path}")
print(f"\nTo evaluate MARCO system:")
print(f"  1. Use the task list from: {marco_config_path}")
print(f"  2. Run MARCO on these tasks")
print(f"  3. Save results to: {CONFIG['results_dir']}/marco_results_{timestamp}.csv")
print(f"  4. Results should include same columns: task_id, grid_accuracy, task_success, time_taken")

## Summary

This notebook evaluated:
- **Base models**: No fine-tuning
- **Stage 1 models**: Fine-tuned for hypothesis generation

Results are saved in `evaluation_results/` with:
- Detailed per-task results (CSV)
- Summary statistics (CSV)
- Comparison plots (PNG)
- MARCO evaluation config (JSON)

### Next Steps

1. **Evaluate MARCO system** using the exported config
2. **Compare all results** (base, stage1, MARCO)
3. **Analyze improvements** from each stage of fine-tuning

### Expected Performance Ranges

Based on typical ARC performance:
- **Base models**: 5-15% grid accuracy
- **Stage 1 (fine-tuned)**: 15-30% grid accuracy
- **MARCO (full system)**: 25-40% grid accuracy (with belief fusion and refinement)